# Customers Data Profiling


## Project Setup


In [2]:
from pathlib import Path
import sys

# Identify the project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Enable imports from src
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Load Customer Data


In [3]:
from pathlib import Path
import pandas as pd

bronze_path = Path.cwd().parent / "data" / "bronze"

# Load the orders
df_customers = pd.read_csv(bronze_path / "olist_customers_dataset.csv")

# Display the first 5 rows
df_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## Check Postal Codes and Missing Values


In [9]:
df_customers = pd.read_csv(
    bronze_path / "olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "string"}
)
zip_codes = df_customers["customer_zip_code_prefix"].astype("string")

# Pad with missing leading zeros
print(df_customers.dtypes)
print(df_customers.isna().sum())

# Check missing or malformed postal codes
invalid_zip_codes = (
    zip_codes.isna()
    | ~zip_codes.str.fullmatch(r"\d{5}").fillna(False)
)

# Display the anomalies
postal_code_anomalies = df_customers[invalid_zip_codes]

print("Nombre de codes postaux invalides :", len(postal_code_anomalies))

display(postal_code_anomalies)

customer_id                         object
customer_unique_id                  object
customer_zip_code_prefix    string[python]
customer_city                       object
customer_state                      object
dtype: object
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Nombre de codes postaux invalides : 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


## Check Duplicates

In [10]:
print(
    "Doublons customer_id :",
    df_customers["customer_id"].duplicated().sum()
)

print(
    "Doublons customer_unique_id :",
    df_customers["customer_unique_id"].duplicated().sum()
)

print(
    "Lignes entièrement dupliquées :",
    df_customers.duplicated().sum()
)

Doublons customer_id : 0
Doublons customer_unique_id : 3345
Lignes entièrement dupliquées : 0


### Check Customer States


In [11]:
print(df_customers["customer_state"].value_counts())

print(
    "Nombre d'États différents :",
    df_customers["customer_state"].nunique()
)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64
Nombre d'États différents : 27


## City Profiling

In [13]:
# Number of distinct cities
print(
    "Nombre de villes différentes :",
    df_customers["customer_city"].nunique()
)

# Check empty cities
empty_cities = (
    df_customers["customer_city"].isna()
    | df_customers["customer_city"].str.strip().eq("")
)

print(
    "Nombre de villes vides :",
    empty_cities.sum()
)

# Display the 10 most frequent cities
print(
    df_customers["customer_city"].value_counts().head(10)
)

Nombre de villes différentes : 4119
Nombre de villes vides : 0
customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64


## Check Whitespace and Capitalization

In [14]:
# Check cities with unnecessary whitespace
cities_with_whitespace = (
    df_customers["customer_city"]
    != df_customers["customer_city"].str.strip()
)

# Check cities containing uppercase letters
cities_with_uppercase = (
    df_customers["customer_city"]
    != df_customers["customer_city"].str.lower()
)

print("Villes avec espaces inutiles :", cities_with_whitespace.sum())
print("Villes contenant des majuscules :", cities_with_uppercase.sum())

Villes avec espaces inutiles : 0
Villes contenant des majuscules : 0
